# LoTTE Corpus Embedding on Google Colab (GPU)

This notebook encodes the 268k passages from the LoTTE dataset using `sentence-transformers/all-MiniLM-L6-v2` on a free Colab T4 GPU (~5–10 minutes).

### Output Files
- `embeddings.npy` (NumPy array, shape `[268893, 384]`, ~413 MB)
- `pids.json` (List of passage IDs matching rows in `embeddings.npy`)

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q sentence-transformers faiss-cpu

## 2. Mount Google Drive

In [ ]:
import importlib

try:
    drive = importlib.import_module("google.colab.drive")
    drive.mount("/content/drive")
except ImportError:
    print("google.colab not available (running outside Colab environment)")

## 3. Extract or Locate the LoTTE Dataset
Ensure `collection.tsv` is accessible (e.g., extracted from `lotte.tar.gz`).

In [ ]:
import os

# Set path to lotte lifestyle dev collection.tsv
# Adjust path if located in your Google Drive or Colab storage:
collection_path = '/content/drive/MyDrive/Quest/lotte/lifestyle/dev/collection.tsv'
if not os.path.exists(collection_path):
    collection_path = '/content/drive/MyDrive/Quest/lotte/lotte/lifestyle/dev/collection.tsv'
if not os.path.exists(collection_path):
    # Fallback: check local Colab or project path
    collection_path = 'lotte/lifestyle/dev/collection.tsv'
if not os.path.exists(collection_path):
    collection_path = 'lotte/lotte/lifestyle/dev/collection.tsv'

print('Collection path:', collection_path)
assert os.path.exists(collection_path), f'Could not find collection.tsv at {collection_path}'

## 4. Load Raw Passages

In [ ]:
pids = []
texts = []

with open(collection_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.rstrip('\n')
        if not line:
            continue
        pid_str, text = line.split('\t', 1)
        pids.append(int(pid_str))
        texts.append(text)

print(f'Loaded {len(texts):,} passages.')

## 5. Encode with SentenceTransformer on GPU
- Model: `all-MiniLM-L6-v2`
- Batch size: `256`
- Max sequence length: `256`
- `normalize_embeddings=True` (unit length for cosine similarity)

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
model.max_seq_length = 256

print('Encoding passages on GPU...')
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)

print(f'Encoded {len(embeddings):,} passages in {time.time()-t0:.1f}s.')
print('Embeddings shape:', embeddings.shape)

## 6. Save Files & Download

In [ ]:
import json
import importlib

output_dir = 'indexes'
os.makedirs(output_dir, exist_ok=True)

emb_file = os.path.join(output_dir, 'embeddings.npy')
pids_file = os.path.join(output_dir, 'pids.json')

np.save(emb_file, embeddings)
with open(pids_file, 'w', encoding='utf-8') as f:
    json.dump(pids, f)

print(f'Saved {emb_file} ({os.path.getsize(emb_file)/(1024*1024):.1f} MB)')
print(f'Saved {pids_file}')

# Optional: Copy directly to Google Drive
drive_target = '/content/drive/MyDrive/Quest/indexes'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_target, exist_ok=True)
    !cp {emb_file} {drive_target}/
    !cp {pids_file} {drive_target}/
    print(f'Copied files to {drive_target}')

# Download to local browser (when running in Colab)
try:
    files = importlib.import_module("google.colab.files")
    files.download(emb_file)
    files.download(pids_file)
except ImportError:
    print(f'Files saved locally: {emb_file}, {pids_file}')